In [2]:
import requests
import base64
import time
import pandas as pd
import json
from pathlib import Path

korpus = pd.read_json("../daten/eigener_korpus.jsonl", lines=True)

<jemalloc>: Unsupported system page size


In [3]:
fabio_korpus = pd.read_json("../daten/fabio_korpus.jsonl", lines=True)
fabio_korpus.head()

,refnr,titel,firma,text,ort
0,15548-1423375-1-S,"(Senior) Consultant- Financial Crime, Complian...",EY Consulting GmbH,Requisition ID: 1610322\n\n\nAre you ready to ...,"Eschborn, Taunus"
1,10000-1205912617-S,IT-Security Consultant (m/w/d) Schwerpunkt VPN,AirITSystems GmbH,## Du kennst dich bestens mit VPN aus?\n\nDu s...,"Langenhagen, Han"
2,14447-43454028-613-S,Project Manager* Global Procurement,HARTING Stiftung & Co.KG,HARTING steht für starke Verbindungen - rund u...,Espelkamp
3,12942-1555106-1-S,Business Analyst (m/w/d) Service Now mobile Apps,Dirk Rossmann GmbH,Unsere Unternehmenszentrale in Burgwedel bei H...,Burgwedel
4,13509-0000210fdcf001-S,Lead IT Expert DevOps & Platform Engineering (...,BWI GmbH,Sorge gemeinsam mit uns für die digitale Zukun...,Hamburg


In [4]:
meine_ids = set(korpus["refnr"])
partner_ids = set(fabio_korpus["refnr"])

schnittmenge = meine_ids.intersection(partner_ids)

print(f"Meine Anzeigen: {len(meine_ids)}")
print(f"Partner Anzeigen: {len(partner_ids)}")
print(f"Gemeinsame Anzeigen: {len(schnittmenge)}")

Meine Anzeigen: 44
Partner Anzeigen: 163
Gemeinsame Anzeigen: 7


In [5]:
gemeinsame_anzeigen = korpus[
    korpus["refnr"].isin(schnittmenge)]

gemeinsame_anzeigen[["refnr", "titel", "firma", "text", "ort"]]

,refnr,titel,firma,text,ort
5,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,Moderne Marineschiffe sind hochkomplexe System...,Bremen
10,15939-BB-633095-7878-7490-S,(Senior) Data Analyst - Lifecycle-Analysen im ...,Rheinmetall AG,Marineschiffe sind hochkomplexe Systeme mit Le...,Bremen
17,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWHAT WE ARE LOOKIN...,Bremen
20,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWOFÜR WIR SIE SUCH...,Bremen
26,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,#### Your Tasks\n\n- Being the project’s focal...,Bremen
36,13999-k53401.30280-S,Business Analyst (m/w/d) im Technologiekonzern,jobtimum GmbH Personalvermittlung,\n\nBusiness Analyst (m/w/d) im Technologiekon...,Bremen
42,15939-BB-632493-7878-9058-S,Praktikant Market Intelligence (m/w/d),Rheinmetall AG,* Unterstützung in den Bereichen Market and Co...,Bremen


In [6]:
# pic 5 random entries from fabio_korpus which aren't in gemeinsame anzeigen and create an dataframe with both
# Anzeigen aus Fabios Korpus, die NICHT bereits in der Schnittmenge sind
nur_fabio = fabio_korpus[
    ~fabio_korpus["refnr"].isin(schnittmenge)
]

print(f"Nur Fabio Anzeigen: {len(nur_fabio)}")
zufaellige_fuenf = nur_fabio.sample(
    n=5,
    random_state=42
)

Nur Fabio Anzeigen: 156


In [7]:
annotations_auswahl = pd.concat(
    [gemeinsame_anzeigen, zufaellige_fuenf],
    ignore_index=True
)
annotations_auswahl.head(14)

,refnr,titel,firma,text,ort,beruf,veroeffentlichung,homeoffice_api,gehalt_api,vertragsdauer_api,raw
0,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,Moderne Marineschiffe sind hochkomplexe System...,Bremen,Data Scientist,2026-03-18,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
1,15939-BB-633095-7878-7490-S,(Senior) Data Analyst - Lifecycle-Analysen im ...,Rheinmetall AG,Marineschiffe sind hochkomplexe Systeme mit Le...,Bremen,Data Scientist,2026-03-18,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
2,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWHAT WE ARE LOOKIN...,Bremen,Fachinformatiker/in - Daten- und Prozessanalyse,2026-04-23,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
3,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWOFÜR WIR SIE SUCH...,Bremen,Machine Learning Engineer,2026-04-23,0.0,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
4,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,#### Your Tasks\n\n- Being the project’s focal...,Bremen,Ingenieur/in - Systems Engineering,2026-03-06,NaN,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
5,13999-k53401.30280-S,Business Analyst (m/w/d) im Technologiekonzern,jobtimum GmbH Personalvermittlung,\n\nBusiness Analyst (m/w/d) im Technologiekon...,Bremen,Business-Analyst/in,2026-04-10,NaN,JAHRESGEHALT,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
6,15939-BB-632493-7878-9058-S,Praktikant Market Intelligence (m/w/d),Rheinmetall AG,* Unterstützung in den Bereichen Market and Co...,Bremen,Data Scientist,2026-04-20,0.0,KEINE_ANGABEN,BEFRISTET,"{'stellenangebotsart': 'PRAKTIKUM_TRAINEE', 's..."
7,13635-7fbe73ac_JB5131141-S,Softwareentwickler:in Machine- / Deep-Learning...,zollsoft GmbH,Wo Du mit anpacken kannst\n • Gleich vom erste...,Hamburg,NaN,NaN,NaN,NaN,NaN,NaN
8,20536-lutif851vc-S,Research Associate / Wissenschaftliche*r Mitar...,Technische Universität Hamburg,Research Associate (m/f/d) - \nWissenschaftl...,Hamburg,NaN,NaN,NaN,NaN,NaN,NaN
9,12826-SA0136034_JB5125696-S,Fachinformatiker für Systemintegration (m/w/d),Piening GmbH Engineering & IT,Wir suchen Dich!\nPiening gehört zu den Top En...,Hamburg,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# save annotations_auswahl to csv

annotations_auswahl.to_csv("../daten/annotaions_auswahl.csv", index=False)

In [9]:
API_KEY = "jobboerse-jobsuche"
BASE_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4"

HEADERS = {
    "X-API-Key": API_KEY
}


def hole_anzeige_details(refnr):
    refnr_encoded = base64.b64encode(
        refnr.encode("utf-8")
    ).decode("utf-8")

    response = requests.get(
        f"{BASE_URL}/jobdetails/{refnr_encoded}",
        headers=HEADERS
    )

    if response.status_code != 200:
        print("Detail fehlgeschlagen:", refnr, response.status_code)
        return None

    detail = response.json()

    anzeige = {
        "refnr": refnr,
        "titel": detail.get("stellenangebotsTitel"),
        "firma": detail.get("firma"),
        "text": detail.get("stellenangebotsBeschreibung"),
        "ort": detail.get("stellenlokationen", [{}])[0].get("adresse", {}).get("ort"),
        "beruf": detail.get("hauptberuf"),
        "veroeffentlichung": detail.get("datumErsteVeroeffentlichung"),
        "homeoffice_api": detail.get("homeofficemoeglich"),
        "gehalt_api": detail.get("verguetungsangabe"),
        "vertragsdauer_api": detail.get("vertragsdauer"),
        "raw": detail
    }

    if not anzeige["text"]:
        print("Keine Stellenbeschreibung:", refnr)
        return None

    return anzeige

In [12]:
fehlende_refnrs = annotations_auswahl.loc[
    annotations_auswahl["raw"].isna(),
    "refnr"
].unique()

fehlende_refnrs

array(['13635-7fbe73ac_JB5131141-S', '20536-lutif851vc-S',
       '12826-SA0136034_JB5125696-S', '13509-00002110865001-S',
       '12862-244829-S'], dtype=object)

In [14]:
nachgeladene_anzeigen = []

for refnr in fehlende_refnrs:
    anzeige = hole_anzeige_details(refnr)

    if anzeige is not None:
        nachgeladene_anzeigen.append(anzeige)

    time.sleep(0.5)

nachgeladen_df = pd.DataFrame(nachgeladene_anzeigen)

nachgeladen_df[["refnr", "titel", "firma", "ort", 'homeoffice_api']]

,refnr,titel,firma,ort,homeoffice_api
0,13635-7fbe73ac_JB5131141-S,Softwareentwickler:in Machine- / Deep-Learning...,zollsoft GmbH,Hamburg,False
1,20536-lutif851vc-S,Research Associate / Wissenschaftliche*r Mitar...,Technische Universität Hamburg,Hamburg,False
2,12826-SA0136034_JB5125696-S,Fachinformatiker für Systemintegration (m/w/d),Piening GmbH Engineering & IT,Hamburg,None
3,13509-00002110865001-S,Senior IT Architect Collaboration und Content ...,BWI GmbH,Berlin,True
4,12862-244829-S,IT Service Desk Agent (m/w/d) 1st Level,AMADEUS FIRE AG,Hamburg,None


In [15]:
nachgeladen_df.head()

,refnr,titel,firma,text,ort,beruf,veroeffentlichung,homeoffice_api,gehalt_api,vertragsdauer_api,raw
0,13635-7fbe73ac_JB5131141-S,Softwareentwickler:in Machine- / Deep-Learning...,zollsoft GmbH,Wo Du mit anpacken kannst\n • Gleich vom erste...,Hamburg,Softwareentwickler/in,2026-04-24,False,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
1,20536-lutif851vc-S,Research Associate / Wissenschaftliche*r Mitar...,Technische Universität Hamburg,Research Associate (m/f/d) - \nWissenschaftl...,Hamburg,Wissenschaftliche/r Mitarbeiter/in,2026-04-27,False,KEINE_ANGABEN,BEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
2,12826-SA0136034_JB5125696-S,Fachinformatiker für Systemintegration (m/w/d),Piening GmbH Engineering & IT,Wir suchen Dich!\nPiening gehört zu den Top En...,Hamburg,Fachinformatiker/in - Systemintegration,2026-04-17,None,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
3,13509-00002110865001-S,Senior IT Architect Collaboration und Content ...,BWI GmbH,Sorge gemeinsam mit uns für die digitale Zukun...,Berlin,Software-Architect,2026-04-28,True,KEINE_ANGABEN,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."
4,12862-244829-S,IT Service Desk Agent (m/w/d) 1st Level,AMADEUS FIRE AG,Seit über 30 Jahren bringen wir qualifizierte ...,Hamburg,Fachinformatiker/in - Systemintegration,2026-04-15,None,JAHRESGEHALT,UNBEFRISTET,"{'stellenangebotsart': 'ARBEIT', 'stellenangeb..."


In [16]:
eigener_korpus = pd.read_json("../daten/eigener_korpus.jsonl", lines=True)

aktualisierter_korpus = pd.concat(
    [eigener_korpus, nachgeladen_df],
    ignore_index=True
)

aktualisierter_korpus = aktualisierter_korpus.drop_duplicates(
    subset="refnr",
    keep="first"
)

aktualisierter_korpus.to_json(
    "../daten/eigener_korpus.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Alter Korpus:", len(eigener_korpus))
print("Nachgeladen:", len(nachgeladen_df))
print("Aktualisierter Korpus:", len(aktualisierter_korpus))

Alter Korpus: 44
Nachgeladen: 5
Aktualisierter Korpus: 49


In [19]:
annotations_auswahl_vollstaendig = aktualisierter_korpus[
    aktualisierter_korpus["refnr"].isin(annotations_auswahl["refnr"])
].copy()

annotaions_auswahl_vollstaendig = annotations_auswahl_vollstaendig.drop_duplicates(
    subset="refnr"
)

annotations_auswahl_vollstaendig[["refnr", "titel", "firma", "text", "ort"]]

,refnr,titel,firma,text,ort
5,15939-BB-633097-7878-9999-S,Data Analyst im Marineschiffbau (m/w/d),Rheinmetall AG,Moderne Marineschiffe sind hochkomplexe System...,Bremen
10,15939-BB-633095-7878-7490-S,(Senior) Data Analyst - Lifecycle-Analysen im ...,Rheinmetall AG,Marineschiffe sind hochkomplexe Systeme mit Le...,Bremen
17,15939-BB-633455-7878-6343-S,Data Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWHAT WE ARE LOOKIN...,Bremen
20,15939-BB-633457-7878-2175-S,MLOps Engineer (m/w/d),Rheinmetall AG,Rheinmetall Digital GmbH\n\nWOFÜR WIR SIE SUCH...,Bremen
26,18777-931781141-S,Requirements Engineer (m/w/d),OHB-System AG,#### Your Tasks\n\n- Being the project’s focal...,Bremen
36,13999-k53401.30280-S,Business Analyst (m/w/d) im Technologiekonzern,jobtimum GmbH Personalvermittlung,\n\nBusiness Analyst (m/w/d) im Technologiekon...,Bremen
42,15939-BB-632493-7878-9058-S,Praktikant Market Intelligence (m/w/d),Rheinmetall AG,* Unterstützung in den Bereichen Market and Co...,Bremen
44,13635-7fbe73ac_JB5131141-S,Softwareentwickler:in Machine- / Deep-Learning...,zollsoft GmbH,Wo Du mit anpacken kannst\n • Gleich vom erste...,Hamburg
45,20536-lutif851vc-S,Research Associate / Wissenschaftliche*r Mitar...,Technische Universität Hamburg,Research Associate (m/f/d) - \nWissenschaftl...,Hamburg
46,12826-SA0136034_JB5125696-S,Fachinformatiker für Systemintegration (m/w/d),Piening GmbH Engineering & IT,Wir suchen Dich!\nPiening gehört zu den Top En...,Hamburg


In [20]:
annotations_auswahl_vollstaendig.to_csv(
    "../daten/annotaions_auswahl.csv",
    index=False
)